# 01 — Explore the raw CFBD pull

Goal: understand each dataset before building anything on it.

1. Row counts by year
2. Missing values per column
3. Position distributions
4. What the recruit `ranking` field actually means
5. How well players match across datasets (recruits / portal → later production)
6. Findings and suggested fixes (not implemented yet)

Everything here reads the cached JSON in `data/raw/`; **no API calls**.

## Setup and load

In [1]:
import glob
import json
import re
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 160)

RAW = Path("..") / "data" / "raw"


def load(subdir, pattern="year=*.json"):
    """Read every cached JSON file in data/raw/<subdir>/ into one flat DataFrame."""
    files = sorted((RAW / subdir).glob(pattern))
    frames = [pd.json_normalize(json.loads(f.read_text(encoding="utf-8"))) for f in files]
    return pd.concat(frames, ignore_index=True)


data = {
    "recruits": load("recruiting/players"),
    "portal": load("player/portal"),
    "ppa_season": load("ppa/players/season"),
    "usage": load("player/usage", "year=*.json"),
    "usage_no_garbage": load("player/usage", "excludeGarbageTime=true_year=*.json"),
    "stats_season": load("stats/player/season"),
    "draft": load("draft/picks"),
}
YEAR_COL = {"recruits": "year", "draft": "year"}  # everything else uses "season"

{name: df.shape for name, df in data.items()}

{'recruits': (37767, 19),
 'portal': (14422, 10),
 'ppa_season': (23179, 22),
 'usage': (23179, 14),
 'usage_no_garbage': (20584, 14),
 'stats_season': (656847, 9),
 'draft': (1551, 24)}

## 1. Row counts by year

In [2]:
counts = pd.DataFrame({
    name: df[YEAR_COL.get(name, "season")].value_counts()
    for name, df in data.items()
}).sort_index().astype("Int64")
counts

,recruits,portal,ppa_season,usage,usage_no_garbage,stats_season,draft
2017,4710,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2018,4348,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2019,4455,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2020,4413,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2021,3364,1770,3317,3317,2897,109056,259
2022,3955,2273,5040,5040,4483,138632,262
2023,4166,2502,4722,4722,4245,132228,259
2024,4236,3378,4481,4481,4009,135294,257
2025,4120,4499,5619,5619,4950,141637,257
2026,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,257


Notes:
- `stats_season` is **long format**: one row per player × stat (e.g. passing YDS), so its
  row count is not a player count. Distinct players per season:

In [3]:
data["stats_season"].groupby("season").playerId.nunique().rename("distinct players")

season
2021    11769
2022    14244
2023    13604
2024    13951
2025    14721
Name: distinct players, dtype: int64

## 2. Missing values per column

In [4]:
def missing_table(df):
    m = df.isna().mean().mul(100).round(1)
    return m[m > 0].sort_values(ascending=False).rename("% missing")

for name, df in data.items():
    print(f"\n=== {name} ({len(df):,} rows) ===")
    t = missing_table(df)
    print(t.to_string() if len(t) else "no missing values")


=== recruits (37,767 rows) ===
athleteId                 40.6
committedTo               17.0
ranking                   12.4
stars                     12.4
rating                    12.4
hometownInfo.fipsCode      1.6
hometownInfo.longitude     1.6
hometownInfo.latitude      1.6
stateProvince              0.6
height                     0.2
weight                     0.2
country                    0.2

=== portal (14,422 rows) ===
rating         54.1
destination    27.5
stars          11.1

=== ppa_season (23,179 rows) ===
averagePPA.rush             41.0
averagePPA.thirdDown        18.1
averagePPA.secondDown       11.9
averagePPA.passingDowns     11.9
averagePPA.firstDown        10.4
averagePPA.pass              8.5
averagePPA.standardDowns     5.2

=== usage (23,179 rows) ===
no missing values

=== usage_no_garbage (20,584 rows) ===
no missing values

=== stats_season (656,847 rows) ===


no missing values

=== draft (1,551 rows) ===
preDraftRanking            6.3
preDraftGrade              6.3
preDraftPositionRanking    6.3
hometownInfo.latitude      5.0
hometownInfo.longitude     5.0
hometownInfo.countyFips    5.0
weight                     3.7
height                     3.7
hometownInfo.state         2.4
hometownInfo.city          1.8
hometownInfo.country       1.8
collegeConference          0.4
collegeAthleteId           0.1


`athleteId` is the recruit's link to on-field data, so its gaps matter most. By class year:

In [5]:
rec = data["recruits"]
rec.groupby("year").athleteId.apply(lambda s: s.isna().mean()).mul(100).round(1).rename("% missing athleteId")

year
2017    67.3
2018    52.5
2019    40.5
2020    40.8
2021    33.1
2022    31.0
2023    35.9
2024    37.0
2025    21.0
Name: % missing athleteId, dtype: float64

In [6]:
# Is missingness related to star rating? (stars itself is missing for unrated recruits)
(rec.assign(stars=rec.stars.astype("Int64").astype("string").fillna("unrated"))
    .groupby("stars").athleteId.apply(lambda s: s.isna().mean()).mul(100).round(1)
    .rename("% missing athleteId"))

stars
2          73.4
3          28.3
4           7.4
5           7.5
unrated    57.8
Name: % missing athleteId, dtype: float64

## 3. Position distributions

Each source uses its own position labels — worth seeing side by side.

In [7]:
POS_COL = {"stats_season": "position", "draft": "position"}
for name, df in data.items():
    col = POS_COL.get(name, "position")
    if name == "stats_season":  # count players, not stat rows
        vc = df.drop_duplicates(["season", "playerId"])[col].value_counts()
    else:
        vc = df[col].value_counts(dropna=False)
    print(f"\n=== {name} ===\n{vc.to_string()}")


=== recruits ===
position
WR      4775
CB      3230
ATH     3202
S       2839
OT      2827
RB      2531
LB      1932
DL      1915
TE      1734
IOL     1731
QB      1343
EDGE    1261
OLB     1221
DT      1073
OG      1052
ILB      884
SDE      841
WDE      788
PRO      738
DUAL     454
K        433
OC       290
P        268
APB      184
LS       184
FB        36
RET        1

=== portal ===
position
WR      2194
CB      1652
DL      1375
LB      1356
S       1310
RB      1084
QB      1071
IOL     1014
OT       927
EDGE     919
TE       737
K        232
P        148
LS       109
ATH       75
PRO       33
OLB       32
ILB       29
DT        29
OG        27
WDE       22
SDE       20
DUAL      19
OC         7
APB        1

=== ppa_season ===
position
WR    9620
RB    6206
TE    3642
QB    3610
FB     101

=== usage ===
position
WR    9620
RB    6206
TE    3642
QB    3610
FB     101

=== usage_no_garbage ===
position
WR    8765
RB    5310
TE    3345
QB    3076
FB      88



=== stats_season ===
position
WR      10847
LB       9683
DB       8207
DL       7670
RB       6961
TE       4221
S        4031
QB       3884
CB       3113
PK       2223
DE       2048
DT       1552
P        1373
OL       1233
LS        653
EDGE      271
FB        141
NT         78
G          42
OT         26
C          14
?           6
ATH         6
PR          4
OLB         1
KR          1

=== draft ===
position
Cornerback            202
Wide Receiver         199
Offensive Tackle      142
Defensive Tackle      141
Safety                120
Running Back          115
Tight End              94
Offensive Guard        93
Defensive Edge         73
Quarterback            67
Linebacker             66
Defensive End          64
Inside Linebacker      52
Center                 49
Outside Linebacker     44
Punter                 12
Place Kicker           11
Fullback                4
Long Snapper            3


In [8]:
# Our scope (QB/RB/WR) by year in the "market" datasets
scope = ["QB", "RB", "WR"]
pd.concat({
    "recruits": pd.crosstab(rec.year, rec.position).reindex(columns=scope + ["ATH"], fill_value=0),
    "portal": pd.crosstab(data["portal"].season, data["portal"].position).reindex(columns=scope, fill_value=0),
}, axis=1)

recruits                portal              
position       QB   RB   WR  ATH     QB     RB     WR
2017            0  310  622  316    NaN    NaN    NaN
2018            2  281  620  313    NaN    NaN    NaN
2019            0  323  530  326    NaN    NaN    NaN
2020            6  281  582  347    NaN    NaN    NaN
2021          182  226  425  264  130.0  145.0  312.0
2022          320  278  456  393  192.0  190.0  345.0
2023          278  282  547  432  188.0  184.0  391.0
2024          263  267  498  458  251.0  267.0  489.0
2025          292  283  495  353  310.0  298.0  657.0

## 4. What does the recruit `ranking` field mean?

Observation that started this: a 2021 recruit with `ranking == 1` has only 3 stars.

In [9]:
r21 = rec[rec.year == 2021]
r21[r21.ranking == 1][["name", "position", "stars", "rating", "ranking", "committedTo", "id"]]

,name,position,stars,rating,ranking,committedTo,id
17926,Marques Buford Jr.,CB,3.0,0.8589,1.0,Nebraska,254930
17927,Quinn Ewers,QB,5.0,1.0000,1.0,Ohio State,115742


**Hypothesis 1: `ranking` is the national rank by `rating`, where tied ratings share a rank.**
Test it by recomputing the rank from `rating` ourselves (`method="min"`: ties share the best rank,
then the next rank skips — like "1, 2, 2, 4").

In [10]:
ranked = rec.dropna(subset=["ranking", "rating"]).copy()
ranked["rank_from_rating"] = ranked.groupby("year").rating.rank(ascending=False, method="min")
ranked["gap"] = ranked.ranking - ranked.rank_from_rating

# Ties are real and common: many 3-stars share identical ratings.
print("2021: rows sharing a ranking value:", r21.ranking.duplicated(keep=False).sum())
r21[r21.rating == 0.8607][["name", "stars", "rating", "ranking"]].head()

2021: rows sharing a ranking value: 2915


,name,stars,rating,ranking
18826,Karate Brenson,3.0,0.8607,894.0
18827,Fred Eaford,3.0,0.8607,894.0
18828,Armorion Smith,3.0,0.8607,894.0
18829,Jay Jones,3.0,0.8607,894.0
18830,Lawson Albright,3.0,0.8607,894.0


In [11]:
# How far is the listed ranking from the rank implied by rating?
ranked["gap_bucket"] = pd.cut(ranked.gap.abs(), [-1, 0, 5, 25, 100, 1e9],
                              labels=["exact", "1-5", "6-25", "26-100", ">100"])
pd.crosstab(ranked.year, ranked.gap_bucket)

gap_bucket,exact,1-5,6-25,26-100,>100
year,,,,,
2017,971,2075,1215,5,12
2018,691,1497,986,690,28
2019,1360,2672,44,1,8
2020,351,2378,1206,0,9
2021,850,1217,653,0,7
2022,1094,1834,423,3,11
2023,1309,1488,656,0,15
2024,1107,2521,65,2,0
2025,375,1454,1812,1,1


In [12]:
# The >100 cases are the genuinely broken rows (e.g. the 3-star ranked #1).
gross = ranked[ranked.gap.abs() > 100]
print(f"{len(gross)} gross mismatches out of {len(ranked):,} ranked recruits")
print("stars:", gross.stars.value_counts().to_dict())
gross.sort_values("gap")[["year", "name", "position", "stars", "rating", "ranking",
                          "rank_from_rating", "id"]].head(10)

91 gross mismatches out of 33,097 ranked recruits
stars: {2.0: 55, 3.0: 35, 4.0: 1}


,year,name,position,stars,rating,ranking,rank_from_rating,id
23,2017,Jaylen Larry,OT,2.0,0.7583,13.0,3931.0,245455
21,2017,Spencer Katoanga,ILB,2.0,0.7644,12.0,3768.0,245461
13530,2020,Miles Frazier,OT,2.0,0.7883,10.0,3352.0,254399
4720,2018,Kenny Terry,SDE,2.0,0.7600,6.0,3266.0,250255
9073,2019,Broc Thompson,WR,2.0,0.7844,8.0,3133.0,259953
18,2017,Fred Cooper,RB,2.0,0.7694,11.0,3075.0,245465
16,2017,Elijah Rivers,OG,2.0,0.7711,10.0,3020.0,245476
15,2017,Malcolm Koonce,SDE,2.0,0.7726,9.0,2964.0,245459
11,2017,Caeleb Washington,SDE,2.0,0.7733,7.0,2919.0,245454
9070,2019,Zach Hoban,K,2.0,0.7922,7.0,2833.0,259986


In [13]:
# rating vs stars: stars are bands of rating, so rating carries strictly more information.
rec.groupby("stars").rating.agg(["count", "min", "median", "max"])

,count,min,median,max
stars,,,,
2.0,8970,0.0700,0.7700,0.7967
3.0,20333,0.7969,0.8411,0.8900
4.0,3490,0.8900,0.9152,0.9832
5.0,306,0.9833,0.9910,1.0000


## 5. Matching players across datasets

Question: what share of recruits and portal entrants can be joined to a **later** PPA or stats line?

Join keys available:

| Dataset | Player key |
|---|---|
| recruits | `athleteId` (often missing), `name`, `committedTo`, `year` |
| portal | **no ID** — only `firstName`, `lastName`, `origin`, `destination`, `season` |
| ppa / usage | `id`, `name`, `team`, `season` |
| stats | `playerId`, `player`, `team`, `season` |
| draft | `collegeAthleteId`, `name`, `collegeTeam` |

Caveat for every match rate below: a missing match is not always a join failure — many recruits
never record a stat (redshirts, depth players, transfers to non-covered levels, quitting).
Restricting to QB/RB/WR, who almost always record stats when they play, makes rates more meaningful.

In [14]:
def norm_name(s):
    """Lowercase, drop punctuation and suffixes: "Marques Buford Jr." -> "marques buford"."""
    s = s.fillna("").str.lower().str.replace(r"[.'’`-]", "", regex=True)
    s = s.str.replace(r"\b(jr|sr|ii|iii|iv|v)\b", "", regex=True)
    return s.str.split().str.join(" ")


ppa = data["ppa_season"]
stats = data["stats_season"]
# One row per player-season-team from the long stats table
stat_players = (stats[["season", "playerId", "player", "team", "position"]]
                .drop_duplicates(["season", "playerId", "team"])
                .rename(columns={"season": "stat_season", "position": "stat_position"}))
stat_players["name_key"] = norm_name(stat_players.player)

ppa_ids = set(ppa.id.astype("string"))
stat_ids = set(stat_players.playerId.astype("string"))
print(f"PPA ids that also appear in stats: {len(ppa_ids & stat_ids) / len(ppa_ids):.1%}")
print("PPA exact duplicate (id, season) rows:", ppa.duplicated(["id", "season"]).sum())

PPA ids that also appear in stats: 99.0%
PPA exact duplicate (id, season) rows: 323


### 5a. Recruits → later production, by `athleteId`

In [15]:
rec = rec.assign(aid=rec.athleteId.astype("string"), name_key=norm_name(rec.name))
rec["id_in_stats"] = rec.aid.isin(stat_ids)
rec["id_in_ppa"] = rec.aid.isin(ppa_ids)

def rates(df, by):
    return df.groupby(by).agg(
        recruits=("id", "size"),
        has_athleteId=("aid", lambda s: s.notna().mean()),
        in_stats=("id_in_stats", "mean"),
        in_ppa=("id_in_ppa", "mean"),
    ).round(3)

rates(rec, "year")

,recruits,has_athleteId,in_stats,in_ppa
year,,,,
2017,4710,0.327,0.117,0.040
2018,4348,0.475,0.271,0.094
2019,4455,0.595,0.395,0.128
2020,4413,0.592,0.420,0.148
2021,3364,0.669,0.502,0.179
2022,3955,0.690,0.500,0.184
2023,4166,0.641,0.458,0.170
2024,4236,0.630,0.398,0.145
2025,4120,0.790,0.288,0.117


In [16]:
# QB/RB/WR only, 2021+ classes (their whole career so far is inside our 2021-2025 data)
rates(rec[rec.position.isin(scope) & (rec.year >= 2021)], ["position"])

,recruits,has_athleteId,in_stats,in_ppa
position,,,,
QB,1335,0.637,0.444,0.421
RB,1336,0.653,0.540,0.516
WR,2421,0.690,0.495,0.444


### 5b. Recruits without `athleteId` → fallback on name + committed team

In [17]:
def name_team_match(left, left_team, left_year):
    """Match on normalized name + team, keeping stat seasons on/after the given year."""
    m = left.merge(stat_players, left_on=["name_key", left_team], right_on=["name_key", "team"],
                   suffixes=("", "_stat"))
    return m[m.stat_season >= m[left_year]]

no_id = rec[rec.aid.isna()]
fallback = name_team_match(no_id, "committedTo", "year")
print(f"Recruits without athleteId: {len(no_id):,}")
print(f"  matched by name + committed team: {fallback.id.nunique():,} "
      f"({fallback.id.nunique() / len(no_id):.1%})")
print("  ambiguous (matched >1 playerId):", (fallback.groupby("id").playerId.nunique() > 1).sum())

# Validate the fallback on recruits that DO have an id: does name+team pick the same player?
check = name_team_match(rec[rec.aid.notna()], "committedTo", "year")
print(f"Validation — name+team agrees with athleteId: {(check.aid == check.playerId).mean():.1%}")

Recruits without athleteId: 15,329
  matched by name + committed team: 2,407 (15.7%)
  ambiguous (matched >1 playerId): 2
Validation — name+team agrees with athleteId: 97.1%


In [18]:
# Combined coverage, QB/RB/WR, 2021+ classes
s = rec[rec.position.isin(scope) & (rec.year >= 2021)].copy()
s["fallback_match"] = s.id.isin(fallback.id)
s["any_match"] = s.id_in_stats | s.fallback_match
s.groupby("position")[["id_in_stats", "fallback_match", "any_match"]].mean().round(3)

,id_in_stats,fallback_match,any_match
position,,,
QB,0.444,0.034,0.478
RB,0.540,0.044,0.584
WR,0.495,0.041,0.536


### 5c. Portal entrants → production before and after the transfer

In [19]:
portal = data["portal"].copy()
portal["name_key"] = norm_name(portal.firstName + " " + portal.lastName)
portal["pid"] = range(len(portal))  # portal rows have no ID, so make one

print(f"Portal rows: {len(portal):,}; no destination: {portal.destination.isna().mean():.1%}")
print("eligibility:", portal.eligibility.value_counts(dropna=False).to_dict())
print("duplicate (name, season) rows:", portal.duplicated(["name_key", "season"]).sum())

Portal rows: 14,422; no destination: 27.5%
eligibility: {'Immediate': 13073, 'TBD': 1074, 'Withdrawn': 256, 'PendingAppeal': 15, 'SittingOne': 4}
duplicate (name, season) rows: 70


In [20]:
moved = portal[portal.destination.notna()]
after = name_team_match(moved, "destination", "season")          # line at new school, same/later season
before = moved.merge(stat_players, left_on=["name_key", "origin"], right_on=["name_key", "team"],
                     suffixes=("", "_stat"))
before = before[before.stat_season < before.season]               # line at old school, earlier season

def portal_rates(df):
    return pd.Series({
        "entrants": len(df),
        "later_line_at_destination": df.pid.isin(after.pid).mean(),
        "prior_line_at_origin": df.pid.isin(before.pid).mean(),
    })

pd.concat({"all": portal_rates(moved),
           **{p: portal_rates(moved[moved.position == p]) for p in scope}}, axis=1).round(3)

,all,QB,RB,WR
entrants,10451.000,807.000,747.000,1562.000
later_line_at_destination,0.672,0.731,0.806,0.770
prior_line_at_origin,0.560,0.616,0.712,0.641


In [21]:
# Why not just match on name alone? Common names collide across teams:
unmatched = moved[~moved.pid.isin(after.pid)]
name_only = unmatched.merge(stat_players, on="name_key", suffixes=("", "_stat"))
name_only = name_only[name_only.stat_season >= name_only.season]
print(f"Unmatched entrants that match a *name only* at some other team: "
      f"{name_only.pid.nunique() / len(unmatched):.1%}")
name_only.drop_duplicates("pid")[["firstName", "lastName", "position", "origin", "destination",
                                  "season", "team", "stat_season"]].head(8)

Unmatched entrants that match a *name only* at some other team: 12.4%


,firstName,lastName,position,origin,destination,season,team,stat_season
0,Jordan,Mitchell,EDGE,Georgia Southern,Texas State,2021,App State,2022
2,Kyron,Cumby,RB,Illinois,Texas Tech,2021,Delaware,2022
5,Damir,Faison,EDGE,East Carolina,Kent State,2021,Towson,2022
6,Robert,Cooper,DL,Boise State,Eastern Washington,2021,Florida State,2021
8,Mark,Williams,CB,Eastern Illinois,UL Monroe,2021,Kent State,2021
10,Luke,Foster,IOL,Presbyterian,Temple,2021,Yale,2024
11,Isaac,Washington,DL,Tennessee,Butler C.C.,2021,Charlotte,2023
14,Martavius,French,LB,Tennessee,Hutchinson C.C.,2021,UTSA,2022


In [22]:
# Team-name consistency between sources (a mismatch silently breaks name+team joins)
teams = set(stat_players.team)
for label, col in [("recruit committedTo", rec.committedTo), ("portal origin", portal.origin),
                   ("portal destination", portal.destination)]:
    missing = col.dropna()[~col.dropna().isin(teams)]
    print(f"{label}: {len(missing) / col.notna().sum():.1%} not found in stats teams; "
          f"top: {missing.value_counts().head(5).to_dict()}")

recruit committedTo: 0.7% not found in stats teams; top: {'Grand Valley State': 147, 'Saint Francis (PA)': 28, 'Southeastern Louisiana': 25, 'St. Thomas': 7, 'Jacksonville': 4}
portal origin: 1.5% not found in stats teams; top: {'Saint Francis (PA)': 13, 'Grand Valley State': 8, 'LIU Post': 8, 'Notre Dame College': 7, 'Northwest Missouri State': 7}
portal destination: 3.2% not found in stats teams; top: {'Texas-Rio Grande Valley': 11, 'Garden City C.C. ': 10, 'Central Oklahoma': 9, 'Navarro College': 9, 'Trinity Valley C.C.': 8}


### 5d. Draft picks → college ID

In [23]:
draft = data["draft"]
draft_ids = draft.collegeAthleteId.astype("Int64").astype("string")
print(f"Draft picks: {len(draft):,}; collegeAthleteId found in stats: {draft_ids.isin(stat_ids).mean():.1%}")
print(f"  ...for picks drafted 2023+ (careers mostly inside our window): "
      f"{draft_ids[draft.year >= 2023].isin(stat_ids).mean():.1%}")
print(f"Draft ids that match a recruit athleteId: {draft_ids.isin(set(rec.aid.dropna())).mean():.1%}")

Draft picks: 1,551; collegeAthleteId found in stats: 73.8%
  ...for picks drafted 2023+ (careers mostly inside our window): 90.0%
Draft ids that match a recruit athleteId: 70.0%



## 6. Findings and suggested fixes

Numbers are from the pull on 2026-11-11. **Fixes below are proposals only — nothing is implemented yet.**

### A. `ranking` = national rank by `rating`, with shared ranks for ties — plus a few broken rows
- `ranking` tracks the composite `rating` almost perfectly (Spearman ≈ −0.99 every year). Equal
  ratings share a rank, then the next rank skips ("1, 2, 2, 4"). 3-star ratings cluster on the same
  values (e.g. every 0.8607 in 2021 is #894), so ties are common.
- **Small drift:** most ranks are within 1–25 places of the rank we recompute from `rating`,
  probably because the source ranked a larger pool than this endpoint returns.
- **91 gross errors** (out of 33,097 ranked): the rating implies about #3,000, but `ranking` says
  #6–13. They're nearly all 2-/3-stars, with IDs in the 245xxx–259xxx block. They look like a rank
  from another list (position or state?) stored in the national field.
  **The 3-star "#1" in 2021 (Marques Buford Jr., rating 0.8589 → about #953) is one of these.**
- **Fix:** use `rating` (continuous) as the market price, not `ranking`. If a rank is needed,
  recompute it from `rating` within each class (and within position). Flag the 91 rows in a
  data-quality column rather than silently dropping them. Separately, check the 2-star minimum
  rating of 0.07, which looks like an outlier.

### B. Position labels differ by source and by year (silent data loss)
- **2017–2020 recruits have no `QB` label**: QBs are `PRO` (pro-style) or `DUAL` (dual-threat),
  and many backs are `APB`. Filtering `position == "QB"` drops every QB from those classes.
- 3,202 recruits are `ATH` (position undecided at signing).
- Stats use `DB`/`DL`/`PK`; draft spells positions out ("Wide Receiver").
- **Fix:** a single position-mapping table (`PRO`/`DUAL` → QB, `APB` → RB, ...). Keep PRO vs DUAL
  as a *feature*, since it's known at signing. For `ATH`, the college position from the first stats
  line can define *which population* a player belongs to. Never use it as a model feature: it's
  information from after signing (leakage).

### C. `athleteId` is missing for 41% of recruits — and not at random
- 73% of **2-stars** are missing it, versus 7% of 4-/5-stars. Coverage also improves over time
  (67% missing in 2017, 21% in 2025).
- That matters for this project: the low-rated recruits are exactly where undervalued players
  would be. Dropping rows without an ID would bias the sample toward "the stars were right."
- **Fallback works:** matching normalized name + committed team (+ stat season ≥ class year)
  agrees with `athleteId` **97.1%** of the time where both exist. It recovers 2,407 more recruits
  (QB/RB/WR coverage for 2021+ classes rises by about 3–4 points).

### D. "No match" is ambiguous: bust or join failure?
- For QB/RB/WR in the 2021+ classes, only about 48–58% of recruits match any stats line, even after
  the fallback. Some of that is real (never played, redshirted, quit, left for a level not covered).
  Some is matching failure (nicknames like "Cam" vs "Cameron", changed teams, missing IDs).
- The model target depends on telling these apart: "0 production" and "unknown" must not be the
  same value.
- **Fix:** record a `match_method` per player (`athlete_id` / `name_team` / `none`), and sample
  the `none` group by hand to estimate how many are truly bad joins.

### E. Portal entrants have no ID at all
- 27.5% have no destination (unsigned or withdrawn). Among those with one, name + destination +
  later season finds a line for **67%** overall (**QB 73%, RB 81%, WR 77%**). Name + origin +
  earlier season finds a prior line for 56% (QB 62%).
- **Matching on name alone is unsafe**: 12.4% of unmatched entrants match *a* same-named player
  somewhere else. Some are real second transfers (Kyron Cumby: Illinois → Texas Tech → Delaware),
  but many are different people with common names (Mark Williams, Robert Cooper).
- 70 duplicate (name, season) rows (re-entries or withdrawals). 256 rows are `Withdrawn`.
- **Transfer "market price" is mostly missing:** portal `rating` is null for 54% and `stars` for 11%.
- **Fix:** match portal → `playerId` using name + team + season window + position group. Then follow
  that `playerId` to get the whole career (before and after, including later transfers). Dedupe
  re-entries by latest `transferDate`, and treat `Withdrawn` separately. Decide how to price
  transfers when `rating` is null (stars only? prior production as the "market's" information?).

### F. Smaller cleanups
- **PPA has 323 exact duplicate `(id, season)` rows** (same team), so drop duplicates before any
  aggregation.
- **Team names:** 3.2% of portal destinations aren't in the stats team list. Most are JUCO/D2
  (legitimately not covered), but there are a few variants and stray whitespace
  (`"Garden City C.C. "`, `"Texas-Rio Grande Valley"`). Strip whitespace and keep a small alias table.
- **`averagePPA.rush` is missing for 41%** (receivers with no rushes). Keep NaN ≠ 0.
- `usage_no_garbage` has 2,595 fewer player-seasons than `usage`: those players *only* played in
  garbage time, which is a useful signal in itself.
- **Draft:** `collegeAthleteId` is found in stats for 74% of picks overall and 90% for 2023+ picks
  (earlier picks played mostly before our 2021 window). 70% match a recruit `athleteId`.

### G. Observation window (censoring)
The 2017–2020 classes are seen only from 2021 on, and the 2025 class has a single season, so raw
"matched / produced" rates aren't comparable across classes. Per-season or first-N-seasons targets
handle this.

### Suggested next step
Build a **player crosswalk** (`src/cfb/players.py`): one row per person, linking recruit `id` /
`athleteId` / portal rows / stats `playerId` / draft `collegeAthleteId`, with `match_method` and
data-quality flags. Write unit tests for the name normalizer and the position map.
